In [ ]:
import numpy as np
import sympy as sp
import pandas as pd

def solve_beam(l1, l2, q1, q2):
    l=l1+l2 #total length
    Mx=sp.symbols('Mx') #create symbol Mx

    #calculate Mx
    Mx=sp.solveset(Mx*l1/3+q1*l1**3/24+Mx*12/3+q2*12**3/24,Mx).args[0]

    #solve equilibrium equations
    Va, Vb1, Vb2, Vc=sp.symbols('Va Vb1 Vb2 Vc')
    Va, Vb1=sp.linsolve([Va+Vb1-q1*l1, Vb1*l1+Mx-(q1*l1**2)/2],
                        (Va, Vb1)).args[0]
    Vc, Vb2=sp.linsolve([Vb2+Vc-q2*l2, Vb2*12+Mx-(q2*l2**2)/2],
                       (Vc, Vb2)).args[0]
    Vb=Vb1+Vb2

    x1=np.arange(0, l1+0.1, 0.1) #create axis x1
    x2=np.arange(0, l2+0.1, 0.1) #cretae axis x2

    beam1=pd.DataFrame({"x":x1}) #create a dataframe for the first span
    beam2=pd.DataFrame({"x":x2}) #create a dataframe for the second span
    
    beam1["M"]=Va*beam1.x-(q1*beam1.x**2)/2 # calculate M and store it
    beam2["M"]=Mx-(q2*beam2.x**2)/2+Vb2*beam2.x # calculate M and store it

    beam1["V"]=Va-q1*beam1.x # calculate V and store it
    beam2["V"]=Vb2-q2*beam2.x # calculate V and store it

    beam2.x=beam2.x+l1 # re-assign x for the second span

    beam=pd.concat([beam1, beam2]) # concatenate the two dataframes

    return(beam) # return the result

header=pd.MultiIndex.from_tuples([(
            ("combo 1", "M"), 
            ("combo 1", "V"),
            ("combo 2", "M"), 
            ("Combo 2", "V")
            ])
combos=pd.DataFrame(columns=header)
combos["x"]=solve_beam(4, 5, 3.2, 4.5)["x"]

result1 = solve_beam(4, 5, 3.2, 4.5)
result2 = solve_beam(4, 5, 4.5, 3.2)

combos[("combo 1", "M")] = result1["M"].to_numpy()
combos[("combo 1", "V")] = result1["V"].to_numpy()

combos[("combo 2", "M")] = result2["M"].to_numpy()
combos[("combo 2", "V")] = result2["V"].to_numpy()

combos.index = result1["x"]
combos.index.name = "x"

combos=combos.astype("float")
combos.head()



    
    



combo 1           combo 2 Combo 2    x combo 2
           M        V        M       V            V
x                                                  
0.0  0.00000  -9.1875  0.00000     NaN  0.0 -2.3625
0.1 -0.93475  -9.5075 -0.25875     NaN  0.1 -2.8125
0.2 -1.90150  -9.8275 -0.56250     NaN  0.2 -3.2625
0.3 -2.90025 -10.1475 -0.91125     NaN  0.3 -3.7125
0.4 -3.93100 -10.4675 -1.30500     NaN  0.4 -4.1625